# CAMELS: Attributes from Stations and Basins
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 03-08-2026<br>

**Introduction:**<br>
This notebook creates the station/basin attributes.

**Ideas:**<br>

In [1]:
import pandas as pd
import geopandas as gpd

from reservoirs_lshm.utils.plots import plot_attributes

from ocab.config import Config

## Configuration

In [2]:
cfg = Config('config_CAMELS_v200.yml')
basin_id_field = 'gauge_id'

# output folders
path_attrs = cfg.path_dataset / 'attributes'
path_plots = cfg.path_plots / 'attributes'
for path in [path_attrs, path_plots]:
    path.mkdir(parents=False, exist_ok=True)
print(f'Attribute tables will be saved in {path_attrs}')

Attribute tables will be saved in /home/casadoj/Data/CAMELS-ES/v2_0_0/attributes


## Ancillary Data

In [3]:
# load basin outlets
outlets = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')
print(f'{len(outlets)} basin outlets')

# load basin polygons
basins = gpd.read_file(cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_3sec.geojson')
basins.columns = basins.columns.str.lower()
basins.set_index('id', inplace=True)
print(f'{len(outlets)} basin polygons')

1116 basin outlets
1116 basin polygons


## Attributes


In [4]:
attributes = []


### Outlets

In [ ]:
cols = ['basin', 'name', 'river', 'province', 'catch_skm', 'geometry']
attr_outlets = outlets[cols]
attr_outlets['lat_wgs84'] = attr_outlets.geometry.y
attr_outlets['lon_wgs84'] = attr_outlets.geometry.x
attr_outlets['x_etrs89'] = attr_outlets.to_crs('epsg:25830').geometry.x
attr_outlets['y_etrs89'] = attr_outlets.to_crs('epsg:25830').geometry.y
attr_outlets.drop(columns='geometry', inplace=True)

# save
attributes.append(attr_outlets)

In [11]:
attr_outlets

,basin,name,province,river,catch_skm,lat_wgs84,lon_wgs84,x_etrs89,y_etrs89
id,,,,,,,,,
1080,CANTABRICO,Andoain,Guipúzcoa,Oria,765.00,43.228338,-2.026305,5.790700e+05,4.786632e+06
1103,CANTABRICO,Alzola,Guipúzcoa,Deba,456.00,43.229455,-2.400994,5.486420e+05,4.786470e+06
1105,CANTABRICO,Ereñozu,Guipúzcoa,Urumea,215.00,43.243559,-1.941541,5.859320e+05,4.788406e+06
1106,CANTABRICO,Endarlaza,Guipúzcoa,Bidasoa,681.00,43.294998,-1.729748,6.030400e+05,4.794358e+06
1107,CANTABRICO,Oyarzun,Guipúzcoa,Oyarzun,38.00,43.298431,-1.876248,5.911510e+05,4.794569e+06
...,...,...,...,...,...,...,...,...,...
10084,CATALUÑA,Riudellots Selva,NaN,Onyar,82.17,41.898318,2.816478,9.825631e+05,4.654874e+06
10085,CATALUÑA,Pont Molins,NaN,Muga,206.55,42.315518,2.917968,9.877650e+05,4.701799e+06
10086,CATALUÑA,Bisbal Emporda,NaN,Daro,143.82,41.964586,3.038219,1.000447e+06,4.663512e+06


### Basins

In [6]:
centroids = basins.to_crs('epsg:25830').centroid
attr_centroids = pd.DataFrame({
    'centroid_x_etrs89': centroids.geometry.x,
    'centroid_y_etrs89': centroids.geometry.y,
    'centroid_lat_wgs84': centroids.to_crs('epsg:4326').geometry.y,
    'centroid_lon_wgs84': centroids.to_crs('epsg:4326').geometry.x,
})

# save
attributes.append(attr_centroids)

### Export

In [7]:
# concatenate all attributes
attrs = pd.concat(attributes, axis=1)
attrs.sort_index(axis=0, inplace=True)
attrs.index = [f'{cfg.prefix}_{ID}' for ID in attrs.index]
attrs.index.name = basin_id_field
print('{0} attributes define the characteristics of {1} catchments'.format(*attrs.shape[::-1]))

# export
attrs.to_csv(path_attrs / 'stations.csv')

12 attributes define the characteristics of 1116 catchments
